In [1]:
# Cell 1 - Imports, configuration, and helper functions
from __future__ import annotations

import asyncio
import json
import os
import statistics
import sys
import time
from pathlib import Path
from typing import Any, Literal

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.core.model_provider import GenerateParams, StructuredParams
from src.core.providers.lmstudio_provider import LMStudioProvider

LM_STUDIO_URL = os.getenv("LM_STUDIO_URL", "http://localhost:1234/v1").rstrip("/")
LM_STUDIO_API_ROOT = LM_STUDIO_URL[:-3] if LM_STUDIO_URL.endswith("/v1") else LM_STUDIO_URL
REQUEST_TIMEOUT_SECONDS = 30
MODEL_OPERATION_TIMEOUT_SECONDS = 180
MODEL_READY_TIMEOUT_SECONDS = 120
MODEL_DOWNLOAD_TIMEOUT_SECONDS = 1800
MODEL_POLL_INTERVAL_SECONDS = 2
LATENCY_BENCHMARK_RUNS = 5
REQUEST_RETRY_LIMIT = 10
REQUEST_RETRY_DELAY_SECONDS = 1.5
NOTEBOOK_HELPER_VERSION = "2026-04-09-v6"
BENCHMARK_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "benchmarks"
BENCHMARK_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_RUN_ID = time.strftime("%Y%m%d_%H%M%S")
LATEST_RESULTS_PATH = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest.json"
RUN_RESULTS_PATH = BENCHMARK_ARTIFACT_DIR / f"02_model_benchmark_{BENCHMARK_RUN_ID}.json"
PRECHECK_PROMPT = "Reply with exactly READY."

print(f"Project root: {PROJECT_ROOT}")
print(f"LM Studio OpenAI endpoint: {LM_STUDIO_URL}")
print(f"LM Studio REST endpoint: {LM_STUDIO_API_ROOT}")

BENCHMARK_MODELS = [
    {
        "name": "Qwen3.5-4B",
        "lmstudio_id": "https://huggingface.co/lmstudio-community/Qwen3.5-4B-GGUF",
        "params": "4B (5B actual)",
        "notes": "English text,Alibaba English text,English text DeltaNet+Attention",
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
    {
        "name": "Gemma-4-E4B-IT",
        "lmstudio_id": "https://huggingface.co/lmstudio-community/gemma-4-E4B-it-GGUF",
        "params": "4B effective",
        "notes": "Google English text,English text,English text,128K English text",
        "extra_body": {},
    },
    {
        "name": "GPT-OSS-Nano",
        "lmstudio_id": "https://huggingface.co/squ11z1/gpt-oss-nano",
        "params": "9B total / 3B active (MoE)",
        "notes": "OpenAI GPT-OSS English text,MoE English text,131K English text",
        "extra_body": {},
    },
]

TEST_PROMPTS = {
    "short_coding": "Write a Python function to check if a number is prime.",
    "short_translation": "Translate to Chinese: 'The quick brown fox jumps over the lazy dog.'",
    "short_factual": "What are the three laws of thermodynamics? Answer in one sentence each.",
    "medium_coding": (
        "Write a Python class called `CSVValidator` with methods: "
        "`__init__(self, filepath: str)` that reads a CSV file, "
        "`validate_headers(self, expected: list[str]) -> bool` that checks column names, "
        "`find_missing_values(self) -> dict[str, int]` that returns column name to missing count mapping, "
        "and `summary(self) -> str` that returns a human-readable validation report. "
        "Include type hints, docstrings, and handle FileNotFoundError."
    ),
    "medium_analysis": (
        "Compare the attention mechanisms in the original Transformer (Vaswani et al., 2017) "
        "with Flash Attention (Dao et al., 2022). Discuss: (1) computational complexity differences, "
        "(2) memory usage patterns, (3) practical impact on training throughput for sequences longer than 4096 tokens, "
        "and (4) under what conditions the original attention might still be preferred."
    ),
    "medium_creative": (
        "You are a senior product manager at a startup. Write a product brief (200 words max) for a new feature: "
        "an AI-powered 'Prompt Optimizer' that takes a user's rough prompt, analyzes it for redundancy and ambiguity, "
        "and outputs a compressed version that achieves better results with fewer tokens. "
        "Include: problem statement, proposed solution, key metrics, and one potential risk."
    ),
    "long_coding": (
        "Design and implement a Python module called `token_budget_manager.py` with the following specifications:\n\n"
        "1. Class `TokenBudget`:\n"
        "   - `__init__(self, total_budget: int, model: str = 'gpt-4o')`\n"
        "   - `allocate(self, section: str, content: str) -> AllocationResult` - allocates tokens from the budget for a named section\n"
        "   - `remaining(self) -> int` - returns remaining token budget\n"
        "   - `report(self) -> BudgetReport` - returns a structured report of all allocations\n"
        "   - `optimize(self, target_reduction: float = 0.2) -> list[Suggestion]` - suggests which sections to compress\n\n"
        "2. Class `AllocationResult(BaseModel)`: section, token_count, budget_remaining, over_budget(bool)\n"
        "3. Class `BudgetReport(BaseModel)`: total_budget, used, remaining, sections(list), utilization_percent\n"
        "4. Class `Suggestion(BaseModel)`: section, current_tokens, suggested_tokens, compression_strategy\n\n"
        "Requirements:\n"
        "- Use tiktoken for token counting\n"
        "- Raise `BudgetExceededError` when allocation exceeds remaining budget (allow force=True to override)\n"
        "- Thread-safe with a simple lock\n"
        "- Include comprehensive docstrings and type hints\n"
        "- The `optimize` method should prioritize sections with the highest token count for compression suggestions"
    ),
    "long_analysis": (
        "You are an AI researcher writing a literature review section. Analyze the following research question and provide a structured response:\n\n"
        "Research Question: How do different prompt compression techniques affect the quality-cost tradeoff in LLM inference?\n\n"
        "Cover these areas:\n"
        "1. Token-level pruning: techniques that remove or replace individual tokens while preserving semantic meaning. "
        "Discuss approaches like LLMLingua, Selective Context, and PCRL. What are their typical compression ratios and quality retention rates?\n"
        "2. Semantic compression: methods that rephrase content more concisely. How do they compare to token-level approaches in terms of meaning preservation?\n"
        "3. Structural optimization: techniques that reorganize prompt structure (positional anchoring, section reordering) to improve attention efficiency without reducing token count.\n"
        "4. Hybrid approaches: systems that combine multiple techniques. What are the theoretical and practical advantages?\n\n"
        "For each area, discuss: (a) key papers and methods, (b) reported compression rates, (c) quality impact metrics, (d) computational overhead of the compression itself.\n\n"
        "Conclude with: what gaps remain in the literature, and what would a practical system need to address?"
    ),
}

STRUCTURED_PROMPT = (
    "Analyze this request for a prompt-optimization assistant: "
    "'Need a faster way to shrink long prompts without losing essential requirements.' "
    "Return the user intent, 3-5 keywords, and an overall complexity level."
)
LATENCY_PROMPT = "Summarize in one sentence why prompt compression can improve inference cost-efficiency."


class SimpleAnalysis(BaseModel):
    intent: str
    keywords: list[str]
    complexity: Literal["low", "medium", "high"]


def _round_or_none(value: float | None, digits: int = 2) -> float | None:
    if value is None:
        return None
    return round(value, digits)


def _preview_text(text: str, limit: int = 300) -> str:
    normalized = " ".join((text or "").split())
    if len(normalized) <= limit:
        return normalized
    return f"{normalized[:limit]}..."


def _normalize_identifier(value: str) -> str:
    return "".join(character for character in value.lower() if character.isalnum())


def _candidate_lookup_keys(model_ref: str) -> list[str]:
    candidates: set[str] = set()
    pieces = [model_ref, model_ref.split("/")[-1], model_ref.rsplit("@", 1)[0]]
    for piece in pieces:
        if not piece:
            continue
        normalized = _normalize_identifier(piece)
        if normalized:
            candidates.add(normalized)

        lower_piece = piece.lower()
        for suffix in ("-gguf", ".gguf", "-mlx", ".mlx"):
            if lower_piece.endswith(suffix):
                trimmed = piece[: -len(suffix)]
                normalized_trimmed = _normalize_identifier(trimmed)
                if normalized_trimmed:
                    candidates.add(normalized_trimmed)

    return sorted(candidates, key=len, reverse=True)


def _is_embedding_model(model_entry: Any) -> bool:
    if isinstance(model_entry, str):
        haystack = model_entry.lower()
    elif isinstance(model_entry, dict):
        haystack = " ".join(
            str(model_entry.get(key, ""))
            for key in ("type", "task", "arch", "id", "model", "model_id", "name", "key")
        ).lower()
    else:
        haystack = str(model_entry).lower()
    return "embedding" in haystack or "embed" in haystack


def _request_json(method: str, url: str, *, timeout: int | float = REQUEST_TIMEOUT_SECONDS, **kwargs: Any) -> Any:
    response = requests.request(method=method, url=url, timeout=timeout, **kwargs)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        body = response.text.strip()
        detail = f" Response body: {body}" if body else ""
        raise requests.HTTPError(f"{exc}.{detail}", response=response) from exc
    if not response.content:
        return {}
    try:
        return response.json()
    except ValueError:
        return {"raw_text": response.text}


def list_all_models() -> list[dict[str, Any]]:
    payload = _request_json("GET", f"{LM_STUDIO_API_ROOT}/api/v1/models")
    models = payload.get("models", []) if isinstance(payload, dict) else []
    print(f"[lmstudio] /api/v1/models returned {len(models)} entries")
    return [entry for entry in models if isinstance(entry, dict)]


def _resolve_downloaded_model_key(model_ref: str, models: list[dict[str, Any]]) -> str | None:
    candidates = _candidate_lookup_keys(model_ref)
    matches: list[tuple[int, str]] = []

    for entry in models:
        model_key = entry.get("key")
        if not isinstance(model_key, str) or not model_key:
            continue

        normalized_key = _normalize_identifier(model_key)
        normalized_display = _normalize_identifier(str(entry.get("display_name", "")))
        publisher = str(entry.get("publisher", ""))
        normalized_publisher_key = _normalize_identifier(f"{publisher}/{model_key}") if publisher else ""

        score = 0
        for candidate in candidates:
            if candidate == normalized_key:
                score = max(score, 400)
            elif normalized_publisher_key and candidate == normalized_publisher_key:
                score = max(score, 390)
            elif candidate == normalized_display:
                score = max(score, 380)
            elif candidate in normalized_key:
                score = max(score, 280)
            elif normalized_key and normalized_key in candidate:
                score = max(score, 260)
            elif candidate in normalized_display:
                score = max(score, 220)
            elif normalized_publisher_key and candidate in normalized_publisher_key:
                score = max(score, 210)

        if score:
            matches.append((score, model_key))

    if not matches:
        return None

    matches.sort(key=lambda item: (-item[0], len(item[1])))
    return matches[0][1]


def list_loaded_models() -> list[dict[str, Any]]:
    loaded_models = []
    for entry in list_all_models():
        loaded_instances = entry.get("loaded_instances") or []
        if loaded_instances:
            loaded_models.append(entry)
    print(f"[lmstudio] found {len(loaded_models)} model entries with loaded instances")
    return loaded_models


def unload_model(model_id: str) -> None:
    print(f"[lmstudio] unloading model instance: {model_id}")
    _request_json(
        "POST",
        f"{LM_STUDIO_API_ROOT}/api/v1/models/unload",
        timeout=MODEL_OPERATION_TIMEOUT_SECONDS,
        json={"instance_id": model_id},
    )


def unload_all_models() -> None:
    unload_targets: list[str] = []
    for entry in list_loaded_models():
        if _is_embedding_model(entry):
            continue
        for instance in entry.get("loaded_instances") or []:
            instance_id = instance.get("id") if isinstance(instance, dict) else None
            if isinstance(instance_id, str) and instance_id:
                unload_targets.append(instance_id)

    if not unload_targets:
        print("[lmstudio] no non-embedding model instances to unload")
        return

    print(f"[lmstudio] unloading {len(unload_targets)} non-embedding model instance(s)")
    for model_id in unload_targets:
        try:
            unload_model(model_id)
        except requests.RequestException as exc:
            print(f"[warning] failed to unload {model_id}: {exc}")


def _wait_for_download(job_id: str, model_ref: str) -> None:
    deadline = time.monotonic() + MODEL_DOWNLOAD_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        status_payload = _request_json(
            "GET",
            f"{LM_STUDIO_API_ROOT}/api/v1/models/download/status/{job_id}",
        )
        status = status_payload.get("status", "unknown")
        downloaded_bytes = status_payload.get("downloaded_bytes")
        total_size_bytes = status_payload.get("total_size_bytes")
        print(
            f"[download] {model_ref}: status={status}, "
            f"downloaded_bytes={downloaded_bytes}, total_size_bytes={total_size_bytes}"
        )

        if status == "completed":
            return
        if status == "failed":
            raise RuntimeError(f"Download failed for {model_ref}: {status_payload}")

        time.sleep(MODEL_POLL_INTERVAL_SECONDS)

    raise RuntimeError(
        f"Timed out after {MODEL_DOWNLOAD_TIMEOUT_SECONDS}s while downloading {model_ref}."
    )


def _ensure_downloaded_model_key(lmstudio_id: str) -> str:
    models_before = list_all_models()
    resolved_key = _resolve_downloaded_model_key(lmstudio_id, models_before)
    if resolved_key:
        print(f"[lmstudio] resolved downloaded model key: {resolved_key}")
        return resolved_key

    print(f"[lmstudio] {lmstudio_id} is not downloaded yet, starting download")
    download_payload = _request_json(
        "POST",
        f"{LM_STUDIO_API_ROOT}/api/v1/models/download",
        timeout=MODEL_OPERATION_TIMEOUT_SECONDS,
        json={"model": lmstudio_id},
    )
    status = download_payload.get("status")
    job_id = download_payload.get("job_id")
    print(f"[download] initial response: {download_payload}")

    if status == "failed":
        raise RuntimeError(f"Download failed for {lmstudio_id}: {download_payload}")
    if job_id:
        _wait_for_download(job_id, lmstudio_id)

    models_after = list_all_models()
    resolved_key = _resolve_downloaded_model_key(lmstudio_id, models_after)
    if resolved_key:
        print(f"[lmstudio] resolved downloaded model key after download: {resolved_key}")
        return resolved_key

    raise RuntimeError(
        f"Download completed but could not resolve a local model key for {lmstudio_id}."
    )


def load_model(lmstudio_id: str) -> str:
    downloaded_key = _ensure_downloaded_model_key(lmstudio_id)
    print(f"[lmstudio] requesting load for {downloaded_key} (from {lmstudio_id})")
    load_payload = _request_json(
        "POST",
        f"{LM_STUDIO_API_ROOT}/api/v1/models/load",
        timeout=MODEL_OPERATION_TIMEOUT_SECONDS,
        json={"model": downloaded_key},
    )
    print(f"[lmstudio] load response: {load_payload}")

    instance_id = load_payload.get("instance_id") if isinstance(load_payload, dict) else None
    if isinstance(instance_id, str) and instance_id:
        return instance_id

    deadline = time.monotonic() + MODEL_READY_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        for entry in list_loaded_models():
            if entry.get("key") != downloaded_key:
                continue
            for instance in entry.get("loaded_instances") or []:
                instance_id = instance.get("id") if isinstance(instance, dict) else None
                if isinstance(instance_id, str) and instance_id:
                    print(f"[lmstudio] resolved instance_id from list_loaded_models: {instance_id}")
                    return instance_id

        print("[lmstudio] waiting for loaded instance to appear in /api/v1/models ...")
        time.sleep(MODEL_POLL_INTERVAL_SECONDS)

    raise RuntimeError(
        f"Timed out after {MODEL_READY_TIMEOUT_SECONDS}s while waiting for {downloaded_key} to expose a loaded instance."
    )


def _build_failed_result(model_config: dict[str, Any], error: str) -> dict[str, Any]:
    return {
        "name": model_config["name"],
        "lmstudio_id": model_config["lmstudio_id"],
        "actual_model_id": None,
        "params": model_config["params"],
        "notes": model_config["notes"],
        "available": False,
        "health": {"available": False, "latency_ms": None, "history": []},
        "prompt_results": [],
        "structured": {"success": False, "content": None, "error": error},
        "latency_runs": [],
        "avg_latency_ms": None,
        "std_latency_ms": None,
        "avg_tokens_per_second": None,
        "response_non_empty_rate": 0.0,
        "structured_success_rate": 0.0,
        "precheck": {
            "ready": False,
            "load_confirmed": False,
            "health_available": False,
            "smoke_test_success": False,
            "health_latency_ms": None,
            "smoke_test_preview": "",
            "smoke_test_completion_tokens": 0,
            "smoke_test_attempts": 0,
            "error": error,
        },
        "load_error": error,
    }


async def run_with_retries(
    label: str,
    operation: Any,
    *,
    attempts: int = REQUEST_RETRY_LIMIT,
    delay_seconds: float = REQUEST_RETRY_DELAY_SECONDS,
) -> tuple[Any, int]:
    last_exc: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            result = await operation()
            if attempt > 1:
                print(f"[retry] {label} succeeded on attempt {attempt}/{attempts}")
            return result, attempt
        except Exception as exc:
            last_exc = exc
            print(f"[retry] {label} failed on attempt {attempt}/{attempts}: {exc}")
            if attempt < attempts:
                await asyncio.sleep(delay_seconds)

    raise RuntimeError(
        f"{label} failed after {attempts} attempts. Last error: {last_exc}"
    ) from last_exc


async def run_single_model_benchmark(
    model_config: dict[str, Any],
    provider: LMStudioProvider,
) -> dict[str, Any]:
    print(f"[health] waiting for {model_config['name']} to become available")
    deadline = time.monotonic() + MODEL_READY_TIMEOUT_SECONDS
    health_history: list[dict[str, Any]] = []
    last_health = None

    while time.monotonic() < deadline:
        health = await provider.health_check()
        last_health = health
        health_history.append(
            {
                "available": health.available,
                "latency_ms": _round_or_none(health.latency_ms),
            }
        )
        if health.available:
            print(
                f"[health] {model_config['name']} ready "
                f"(latency={health.latency_ms:.2f} ms)"
            )
            break
        print(f"[health] not ready yet, retrying in {MODEL_POLL_INTERVAL_SECONDS}s ...")
        await asyncio.sleep(MODEL_POLL_INTERVAL_SECONDS)

    if last_health is None or not last_health.available:
        raise RuntimeError(
            f"Model {model_config['name']} did not pass health_check within {MODEL_READY_TIMEOUT_SECONDS}s."
        )

    prompt_results: list[dict[str, Any]] = []
    for prompt_name, prompt in TEST_PROMPTS.items():
        prompt_length = prompt_name.split("_", 1)[0]
        max_tokens = {
            "short": 256,
            "medium": 512,
            "long": 1024,
        }[prompt_length]

        print(f"[generate] {model_config['name']} -> {prompt_name}")
        started = time.perf_counter()
        try:
            result, attempts_used = await run_with_retries(
                label=f"{model_config['name']} prompt {prompt_name}",
                operation=lambda: provider.generate(
                    GenerateParams(
                        prompt=prompt,
                        max_tokens=max_tokens,
                        temperature=0.2,
                    )
                ),
            )
            latency_ms = (time.perf_counter() - started) * 1000
            tokens_per_second = (
                result.usage.completion_tokens / (latency_ms / 1000)
                if latency_ms > 0 and result.usage.completion_tokens > 0
                else 0.0
            )
            prompt_row = {
                "prompt_name": prompt_name,
                "prompt_length": prompt_length,
                "prompt": prompt,
                "response_text": result.text,
                "response_preview": _preview_text(result.text),
                "response_non_empty": bool(result.text.strip()),
                "prompt_tokens": result.usage.prompt_tokens,
                "completion_tokens": result.usage.completion_tokens,
                "latency_ms": round(latency_ms, 2),
                "tokens_per_second": round(tokens_per_second, 2),
                "attempts_used": attempts_used,
                "error": "",
            }
        except Exception as exc:
            latency_ms = (time.perf_counter() - started) * 1000
            prompt_row = {
                "prompt_name": prompt_name,
                "prompt_length": prompt_length,
                "prompt": prompt,
                "response_text": "",
                "response_preview": "",
                "response_non_empty": False,
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "latency_ms": round(latency_ms, 2),
                "tokens_per_second": 0.0,
                "attempts_used": REQUEST_RETRY_LIMIT,
                "error": str(exc),
            }

        prompt_results.append(prompt_row)
        print(
            f"[generate] {prompt_name}: latency={prompt_row['latency_ms']} ms, "
            f"completion_tokens={prompt_row['completion_tokens']}, "
            f"tokens/s={prompt_row['tokens_per_second']}, "
            f"attempts={prompt_row['attempts_used']}, "
            f"non_empty={prompt_row['response_non_empty']}"
        )
        if prompt_row["error"]:
            print(f"[warning] prompt failed: {prompt_row['error']}")

    print(f"[structured] {model_config['name']} -> SimpleAnalysis")
    try:
        structured_result, structured_attempts_used = await run_with_retries(
            label=f"{model_config['name']} structured generation",
            operation=lambda: provider.generate_structured(
                StructuredParams(
                    prompt=STRUCTURED_PROMPT,
                    schema=SimpleAnalysis.model_json_schema(),
                    max_tokens=220,
                ),
                response_type=SimpleAnalysis,
            ),
        )
        structured_payload = structured_result.model_dump()
        structured_success = True
        structured_error = ""
        print(
            f"[structured] success after {structured_attempts_used} attempt(s): "
            f"{json.dumps(structured_payload, ensure_ascii=False)}"
        )
    except Exception as exc:
        structured_payload = None
        structured_success = False
        structured_attempts_used = REQUEST_RETRY_LIMIT
        structured_error = str(exc)
        print(f"[warning] structured output failed: {structured_error}")

    latency_runs: list[dict[str, Any]] = []
    print(f"[latency] running {LATENCY_BENCHMARK_RUNS} repeated requests for {model_config['name']}")
    for run_index in range(1, LATENCY_BENCHMARK_RUNS + 1):
        started = time.perf_counter()
        try:
            result, attempts_used = await run_with_retries(
                label=f"{model_config['name']} latency run {run_index}",
                operation=lambda: provider.generate(
                    GenerateParams(
                        prompt=LATENCY_PROMPT,
                        max_tokens=96,
                        temperature=0.0,
                    )
                ),
            )
            latency_ms = (time.perf_counter() - started) * 1000
            tokens_per_second = (
                result.usage.completion_tokens / (latency_ms / 1000)
                if latency_ms > 0 and result.usage.completion_tokens > 0
                else 0.0
            )
            run_row = {
                "run": run_index,
                "latency_ms": round(latency_ms, 2),
                "completion_tokens": result.usage.completion_tokens,
                "tokens_per_second": round(tokens_per_second, 2),
                "attempts_used": attempts_used,
                "error": "",
            }
        except Exception as exc:
            latency_ms = (time.perf_counter() - started) * 1000
            run_row = {
                "run": run_index,
                "latency_ms": round(latency_ms, 2),
                "completion_tokens": 0,
                "tokens_per_second": 0.0,
                "attempts_used": REQUEST_RETRY_LIMIT,
                "error": str(exc),
            }

        latency_runs.append(run_row)
        print(
            f"[latency] run={run_index} latency={run_row['latency_ms']} ms "
            f"completion_tokens={run_row['completion_tokens']} "
            f"tokens/s={run_row['tokens_per_second']} "
            f"attempts={run_row['attempts_used']}"
        )
        if run_row["error"]:
            print(f"[warning] latency run failed: {run_row['error']}")

    successful_latency_runs = [row for row in latency_runs if not row["error"]]
    latency_values = [row["latency_ms"] for row in successful_latency_runs]
    throughput_values = [row["tokens_per_second"] for row in successful_latency_runs]

    response_non_empty_rate = (
        sum(1 for row in prompt_results if row["response_non_empty"]) / len(prompt_results)
        if prompt_results
        else 0.0
    )

    return {
        "name": model_config["name"],
        "lmstudio_id": model_config["lmstudio_id"],
        "actual_model_id": provider.model,
        "params": model_config["params"],
        "notes": model_config["notes"],
        "available": True,
        "health": {
            "available": True,
            "latency_ms": _round_or_none(last_health.latency_ms if last_health else None),
            "history": health_history,
        },
        "prompt_results": prompt_results,
        "structured": {
            "success": structured_success,
            "content": structured_payload,
            "attempts_used": structured_attempts_used,
            "error": structured_error,
        },
        "latency_runs": latency_runs,
        "avg_latency_ms": _round_or_none(statistics.fmean(latency_values) if latency_values else None),
        "std_latency_ms": _round_or_none(
            statistics.pstdev(latency_values) if len(latency_values) > 1 else 0.0 if latency_values else None
        ),
        "avg_tokens_per_second": _round_or_none(statistics.fmean(throughput_values) if throughput_values else None),
        "response_non_empty_rate": round(response_non_empty_rate, 3),
        "structured_success_rate": 1.0 if structured_success else 0.0,
        "load_error": "",
    }


def _extract_loaded_instance_ids(entry: dict[str, Any]) -> list[str]:
    instance_ids: list[str] = []
    for instance in entry.get("loaded_instances") or []:
        instance_id = instance.get("id") if isinstance(instance, dict) else None
        if isinstance(instance_id, str) and instance_id:
            instance_ids.append(instance_id)
    return instance_ids


def _find_loaded_entry_for_instance(instance_id: str) -> dict[str, Any] | None:
    for entry in list_loaded_models():
        if instance_id in _extract_loaded_instance_ids(entry):
            return entry
    return None


async def run_model_precheck(
    model_config: dict[str, Any],
    provider: LMStudioProvider,
    instance_id: str,
) -> dict[str, Any]:
    precheck = {
        "ready": False,
        "load_confirmed": False,
        "health_available": False,
        "smoke_test_success": False,
        "health_latency_ms": None,
        "smoke_test_preview": "",
        "smoke_test_completion_tokens": 0,
        "smoke_test_attempts": 0,
        "error": "",
    }

    print(f"[precheck] confirming loaded instance for {model_config['name']}: {instance_id}")
    deadline = time.monotonic() + MODEL_READY_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        loaded_entry = _find_loaded_entry_for_instance(instance_id)
        if loaded_entry is not None:
            precheck["load_confirmed"] = True
            print(
                f"[precheck] load confirmed: key={loaded_entry.get('key')} "
                f"instances={_extract_loaded_instance_ids(loaded_entry)}"
            )
            break
        print(f"[precheck] waiting for instance {instance_id} to appear in loaded models ...")
        await asyncio.sleep(MODEL_POLL_INTERVAL_SECONDS)

    if not precheck["load_confirmed"]:
        precheck["error"] = f"Loaded instance {instance_id} did not appear in /api/v1/models."
        return precheck

    health = await provider.health_check()
    precheck["health_available"] = health.available
    precheck["health_latency_ms"] = _round_or_none(health.latency_ms)
    print(
        f"[precheck] health: available={health.available} latency_ms={precheck['health_latency_ms']}"
    )
    if not health.available:
        precheck["error"] = "Health check failed during precheck."
        return precheck

    try:
        smoke_result, smoke_attempts_used = await run_with_retries(
            label=f"{model_config['name']} precheck smoke test",
            operation=lambda: provider.generate(
                GenerateParams(
                    prompt=PRECHECK_PROMPT,
                    max_tokens=64,
                    temperature=0.0,
                )
            ),
        )
        smoke_preview = _preview_text(smoke_result.text, limit=80)
        precheck["smoke_test_preview"] = smoke_preview
        precheck["smoke_test_completion_tokens"] = smoke_result.usage.completion_tokens
        precheck["smoke_test_attempts"] = smoke_attempts_used
        precheck["smoke_test_success"] = bool(smoke_result.text.strip())
        print(
            f"[precheck] smoke test: success={precheck['smoke_test_success']} "
            f"attempts={precheck['smoke_test_attempts']} "
            f"preview={smoke_preview!r} completion_tokens={precheck['smoke_test_completion_tokens']}"
        )
    except Exception as exc:
        precheck["smoke_test_attempts"] = REQUEST_RETRY_LIMIT
        precheck["error"] = f"Smoke test failed: {exc}"
        print(f"[warning] precheck smoke test failed: {precheck['error']}")
        return precheck

    if not precheck["smoke_test_success"]:
        precheck["error"] = "Smoke test returned an empty response."
        return precheck

    precheck["ready"] = True
    return precheck


def build_precheck_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "precheck_ready": (result.get("precheck") or {}).get("ready", False),
                "load_confirmed": (result.get("precheck") or {}).get("load_confirmed", False),
                "health_available": (result.get("precheck") or {}).get("health_available", False),
                "smoke_test_success": (result.get("precheck") or {}).get("smoke_test_success", False),
                "health_latency_ms": (result.get("precheck") or {}).get("health_latency_ms"),
                "smoke_test_attempts": (result.get("precheck") or {}).get("smoke_test_attempts", 0),
                "smoke_test_preview": (result.get("precheck") or {}).get("smoke_test_preview", ""),
                "precheck_error": (result.get("precheck") or {}).get("error", ""),
            }
            for result in results
        ]
    )


def build_run_summary_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "actual_model_id": result["actual_model_id"],
                "available": result["available"],
                "precheck_ready": (result.get("precheck") or {}).get("ready", False),
                "avg_latency_ms": result["avg_latency_ms"],
                "avg_tokens_per_second": result["avg_tokens_per_second"],
                "structured_success_rate": result["structured_success_rate"],
                "response_non_empty_rate": result["response_non_empty_rate"],
                "max_prompt_attempts": max(
                    [row.get("attempts_used", 1) for row in result.get("prompt_results", [])],
                    default=0,
                ),
                "structured_attempts_used": (result.get("structured") or {}).get("attempts_used", 0),
                "load_error": result["load_error"],
            }
            for result in results
        ]
    )


def build_quality_results_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    quality_rows: list[dict[str, Any]] = []
    for result in results:
        for prompt_result in result.get("prompt_results", []):
            quality_rows.append(
                {
                    "prompt_name": prompt_result["prompt_name"],
                    "model_name": result["name"],
                    "response_preview": prompt_result["response_preview"] or "<empty>",
                    "completion_tokens": prompt_result["completion_tokens"],
                    "tokens_per_second": prompt_result["tokens_per_second"],
                    "attempts_used": prompt_result.get("attempts_used", 1),
                    "error": prompt_result["error"],
                }
            )
    return pd.DataFrame(quality_rows)


def build_latency_runs_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "run": row["run"],
                "latency_ms": row["latency_ms"],
                "completion_tokens": row["completion_tokens"],
                "tokens_per_second": row["tokens_per_second"],
                "attempts_used": row.get("attempts_used", 1),
                "error": row["error"],
            }
            for result in results
            for row in result.get("latency_runs", [])
        ]
    )


def build_latency_summary_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "available": result["available"],
                "precheck_ready": (result.get("precheck") or {}).get("ready", False),
                "avg_latency_ms": result["avg_latency_ms"],
                "std_latency_ms": result["std_latency_ms"],
                "avg_tokens_per_second": result["avg_tokens_per_second"],
            }
            for result in results
        ]
    )


def build_prompt_length_latency_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    prompt_latency_source = pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "prompt_name": prompt_result["prompt_name"],
                "prompt_length": prompt_result["prompt_length"],
                "latency_ms": prompt_result["latency_ms"],
                "tokens_per_second": prompt_result["tokens_per_second"],
            }
            for result in results
            for prompt_result in result.get("prompt_results", [])
        ]
    )
    if prompt_latency_source.empty:
        return prompt_latency_source

    prompt_length_latency_df = (
        prompt_latency_source.groupby(["prompt_length", "model_name"], as_index=False)
        .agg(
            avg_latency_ms=("latency_ms", "mean"),
            std_latency_ms=("latency_ms", "std"),
            avg_tokens_per_second=("tokens_per_second", "mean"),
        )
        .fillna({"std_latency_ms": 0.0})
    )
    prompt_length_latency_df[["avg_latency_ms", "std_latency_ms", "avg_tokens_per_second"]] = (
        prompt_length_latency_df[["avg_latency_ms", "std_latency_ms", "avg_tokens_per_second"]].round(2)
    )
    return prompt_length_latency_df


def build_structured_results_df(results: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["name"],
                "available": result["available"],
                "precheck_ready": (result.get("precheck") or {}).get("ready", False),
                "structured_success": result["structured"]["success"],
                "returned_content": (
                    json.dumps(result["structured"]["content"], ensure_ascii=False)
                    if result["structured"]["content"] is not None
                    else ""
                ),
                "attempts_used": (result.get("structured") or {}).get("attempts_used", 0),
                "error": result["structured"]["error"] or result["load_error"],
            }
            for result in results
        ]
    )


def save_results_snapshot(results: list[dict[str, Any]]) -> dict[str, str]:
    payload = {
        "helper_version": NOTEBOOK_HELPER_VERSION,
        "run_id": BENCHMARK_RUN_ID,
        "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "results": results,
    }
    LATEST_RESULTS_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    RUN_RESULTS_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    summary_csv = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest_summary.csv"
    precheck_csv = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest_precheck.csv"
    quality_csv = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest_quality.csv"
    latency_csv = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest_latency.csv"
    structured_csv = BENCHMARK_ARTIFACT_DIR / "02_model_benchmark_latest_structured.csv"

    build_run_summary_df(results).to_csv(summary_csv, index=False)
    build_precheck_df(results).to_csv(precheck_csv, index=False)
    build_quality_results_df(results).to_csv(quality_csv, index=False)
    build_latency_summary_df(results).to_csv(latency_csv, index=False)
    build_structured_results_df(results).to_csv(structured_csv, index=False)

    return {
        "latest_json": str(LATEST_RESULTS_PATH),
        "run_json": str(RUN_RESULTS_PATH),
        "summary_csv": str(summary_csv),
        "precheck_csv": str(precheck_csv),
        "quality_csv": str(quality_csv),
        "latency_csv": str(latency_csv),
        "structured_csv": str(structured_csv),
    }


def load_saved_results(path: Path | None = None, *, require: bool = True) -> list[dict[str, Any]] | None:
    target_path = path or LATEST_RESULTS_PATH
    if not target_path.exists():
        if require:
            raise FileNotFoundError(f"Saved benchmark results not found: {target_path}")
        return None

    payload = json.loads(target_path.read_text(encoding="utf-8"))
    active_model_names = {model["name"] for model in BENCHMARK_MODELS}
    return [
        result
        for result in payload.get("results", [])
        if result.get("name") in active_model_names
    ]


def ensure_results_loaded() -> list[dict[str, Any]]:
    global ALL_RESULTS
    if "ALL_RESULTS" in globals() and ALL_RESULTS:
        return ALL_RESULTS

    loaded_results = load_saved_results(require=False)
    if not loaded_results:
        raise RuntimeError(
            "No benchmark results are loaded in memory and no saved snapshot was found. Run Cell 2 first."
        )

    ALL_RESULTS = loaded_results
    print(f"[cache] loaded {len(ALL_RESULTS)} saved result(s) from {LATEST_RESULTS_PATH}")
    return ALL_RESULTS


Project root: /Users/corey/Desktop/JHU/Generative AI/TokenLess
LM Studio OpenAI endpoint: http://localhost:1234/v1
LM Studio REST endpoint: http://localhost:1234


In [2]:
# Cell 2 - Sequential benchmark execution
ALL_RESULTS: list[dict[str, Any]] = []
FORCE_RERUN_MODELS: set[str] = set()
# Example: FORCE_RERUN_MODELS = {"GPT-OSS-Nano"}
RESUME_FROM_SAVED_RESULTS = True
REUSE_SAVED_SUCCESSFUL_RESULTS_ONLY = True
RUN_PRECHECK_BEFORE_BENCHMARK = True

print(f"Notebook helper version: {NOTEBOOK_HELPER_VERSION}")
print(f"First benchmark model ref: {BENCHMARK_MODELS[0]['lmstudio_id']}")
print(f"Artifacts directory: {BENCHMARK_ARTIFACT_DIR}")
print(f"Request retry limit: {REQUEST_RETRY_LIMIT} (delay={REQUEST_RETRY_DELAY_SECONDS}s)")

saved_results_by_name: dict[str, dict[str, Any]] = {}
if RESUME_FROM_SAVED_RESULTS:
    cached_results = load_saved_results(require=False) or []
    for cached_result in cached_results:
        if REUSE_SAVED_SUCCESSFUL_RESULTS_ONLY and not cached_result.get("available"):
            continue
        model_name = cached_result.get("name")
        if isinstance(model_name, str) and model_name:
            saved_results_by_name[model_name] = cached_result
    if saved_results_by_name:
        print(
            f"[cache] found {len(saved_results_by_name)} reusable saved result(s) in {LATEST_RESULTS_PATH}"
        )

for model in BENCHMARK_MODELS:
    model_name = model["name"]
    if model_name in saved_results_by_name and model_name not in FORCE_RERUN_MODELS:
        print(f"[cache] reusing saved benchmark result for {model_name}")
        ALL_RESULTS.append(saved_results_by_name[model_name])
        continue

    print("\n" + "=" * 60)
    print(f"Testing: {model['name']} ({model['params']})")
    print("=" * 60)

    actual_model_id = None
    precheck = None
    try:
        unload_all_models()
        actual_model_id = load_model(model["lmstudio_id"])
        provider = LMStudioProvider(
            model=actual_model_id,
            base_url=LM_STUDIO_URL,
            extra_body=model.get("extra_body") or None,
        )

        if RUN_PRECHECK_BEFORE_BENCHMARK:
            precheck = await run_model_precheck(model, provider, actual_model_id)
            if not precheck["ready"]:
                raise RuntimeError(f"Precheck failed: {precheck['error']}")

        results = await run_single_model_benchmark(model, provider)
        results["precheck"] = precheck or results.get("precheck")
        ALL_RESULTS.append(results)
        saved_paths = save_results_snapshot(ALL_RESULTS)
        print(
            f"[done] {model['name']} avg_latency_ms={results['avg_latency_ms']} "
            f"avg_tokens_per_second={results['avg_tokens_per_second']} "
            f"structured_success_rate={results['structured_success_rate']} "
            f"response_non_empty_rate={results['response_non_empty_rate']}"
        )
        print(f"[save] snapshot updated: {saved_paths['latest_json']}")
    except Exception as exc:
        warning = f"{type(exc).__name__}: {exc}"
        print(f"[warning] skipping {model['name']} because setup or benchmark failed: {warning}")
        failed_result = _build_failed_result(model, warning)
        failed_result["actual_model_id"] = actual_model_id
        if precheck is not None:
            failed_result["precheck"] = precheck
        ALL_RESULTS.append(failed_result)
        saved_paths = save_results_snapshot(ALL_RESULTS)
        print(f"[save] failure snapshot updated: {saved_paths['latest_json']}")
    finally:
        if actual_model_id:
            try:
                unload_model(actual_model_id)
            except Exception as unload_exc:
                print(f"[warning] direct unload failed for {actual_model_id}: {unload_exc}")
                try:
                    unload_all_models()
                except Exception as fallback_exc:
                    print(f"[warning] fallback unload_all_models also failed: {fallback_exc}")

PRECHECK_DF = build_precheck_df(ALL_RESULTS)
RUN_SUMMARY_DF = build_run_summary_df(ALL_RESULTS)
SAVED_PATHS = save_results_snapshot(ALL_RESULTS)

print("Precheck summary")
display(PRECHECK_DF)
print("Benchmark summary")
display(RUN_SUMMARY_DF)
print("Saved artifacts")
print(json.dumps(SAVED_PATHS, ensure_ascii=False, indent=2))

Notebook helper version: 2026-04-09-v5
First benchmark model ref: https://huggingface.co/lmstudio-community/Qwen3.5-4B-GGUF
Artifacts directory: /Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks

Testing: Qwen3.5-4B (4B (5B actual))
[lmstudio] /api/v1/models returned 6 entries
[lmstudio] found 0 model entries with loaded instances
[lmstudio] no non-embedding model instances to unload
[lmstudio] /api/v1/models returned 6 entries
[lmstudio] resolved downloaded model key: qwen3.5-4b
[lmstudio] requesting load for qwen3.5-4b (from https://huggingface.co/lmstudio-community/Qwen3.5-4B-GGUF)
[lmstudio] load response: {'type': 'llm', 'instance_id': 'qwen3.5-4b', 'load_time_seconds': 7.37, 'status': 'loaded'}
[precheck] confirming loaded instance for Qwen3.5-4B: qwen3.5-4b
[lmstudio] /api/v1/models returned 6 entries
[lmstudio] found 1 model entries with loaded instances
[precheck] load confirmed: key=qwen3.5-4b instances=['qwen3.5-4b']
[precheck] health: available=True late

LM Studio text generation failed for model 'gpt-oss-nano' with status 400.
Traceback (most recent call last):
  File "/Users/corey/Desktop/JHU/Generative AI/TokenLess/src/core/providers/lmstudio_provider.py", line 73, in generate
    completion = await self._client.chat.completions.create(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/opt/anaconda3/lib/python3.13/site-packages/openai/resources/chat/completions/completions.py", line 2714, in create
    return await self._post(
           ^^^^^^^^^^^^^^^^^
    ...<49 lines>...
    )
    ^
  File "/opt/anaconda3/lib/python3.13/site-packages/openai/_base_client.py", line 1913, in post
    return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/openai/_base_client.py", line 1698, in request
    raise self._make_status_error_from_r

[generate] short_translation: latency=3208.52 ms, completion_tokens=0, tokens/s=0.0, non_empty=False
[warning] prompt failed: LM Studio text generation failed for model 'gpt-oss-nano' with status 400.
[generate] GPT-OSS-Nano -> short_factual
[generate] short_factual: latency=18819.36 ms, completion_tokens=256, tokens/s=13.6, non_empty=True
[generate] GPT-OSS-Nano -> medium_coding
[generate] medium_coding: latency=37441.47 ms, completion_tokens=512, tokens/s=13.67, non_empty=True
[generate] GPT-OSS-Nano -> medium_analysis
[generate] medium_analysis: latency=37212.55 ms, completion_tokens=512, tokens/s=13.76, non_empty=True
[generate] GPT-OSS-Nano -> medium_creative
[generate] medium_creative: latency=37507.25 ms, completion_tokens=512, tokens/s=13.65, non_empty=True
[generate] GPT-OSS-Nano -> long_coding
[generate] long_coding: latency=75093.34 ms, completion_tokens=1024, tokens/s=13.64, non_empty=True
[generate] GPT-OSS-Nano -> long_analysis
[generate] long_analysis: latency=63851.06 m

,model_name,precheck_ready,load_confirmed,health_available,smoke_test_success,health_latency_ms,smoke_test_preview,precheck_error
0,Qwen3.5-4B,True,True,True,True,169.78,Thinking Process: 1. **Analyze the Request:** ...,
1,Gemma-4-E4B-IT,True,True,True,True,4.74,READY,
2,GPT-OSS-Nano,True,True,True,True,10.87,"READY The user says ""Reply with exactly READY....",


Benchmark summary


,model_name,actual_model_id,available,precheck_ready,avg_latency_ms,avg_tokens_per_second,structured_success_rate,response_non_empty_rate,load_error
0,Qwen3.5-4B,qwen3.5-4b,True,True,7661.10,12.54,1.0,1.000,
1,Gemma-4-E4B-IT,gemma-4-e4b-it,True,True,2304.60,11.28,1.0,1.000,
2,GPT-OSS-Nano,gpt-oss-nano,True,True,6364.74,15.11,1.0,0.875,


Saved artifacts
{
  "latest_json": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest.json",
  "run_json": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_20260409_163640.json",
  "summary_csv": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest_summary.csv",
  "precheck_csv": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest_precheck.csv",
  "quality_csv": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest_quality.csv",
  "latency_csv": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest_latency.csv",
  "structured_csv": "/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest_structured.csv"
}


In [3]:
# Cell 3 - Generation quality comparison
ALL_RESULTS = ensure_results_loaded()
QUALITY_RESULTS_DF = build_quality_results_df(ALL_RESULTS)

if QUALITY_RESULTS_DF.empty:
    print("No quality rows were collected.")
else:
    QUALITY_TEXT_DF = (
        QUALITY_RESULTS_DF.pivot(index="prompt_name", columns="model_name", values="response_preview")
        .reindex(list(TEST_PROMPTS.keys()))
        .fillna("SKIPPED")
    )

    metric_rows = QUALITY_RESULTS_DF.copy()
    metric_rows["metric"] = metric_rows.apply(
        lambda row: (
            f"ERROR: {row['error']}"
            if row["error"]
            else (
                f"{int(row['completion_tokens'])} tok | {row['tokens_per_second']:.2f} tok/s | "
                f"attempts={int(row.get('attempts_used', 1))}"
            )
        ),
        axis=1,
    )
    QUALITY_METRICS_DF = (
        metric_rows.pivot(index="prompt_name", columns="model_name", values="metric")
        .reindex(list(TEST_PROMPTS.keys()))
        .fillna("SKIPPED")
    )

    print("Response preview (first 300 chars)")
    display(QUALITY_TEXT_DF)

    print("Completion tokens and throughput")
    display(QUALITY_METRICS_DF)

Response preview (first 300 chars)


model_name,GPT-OSS-Nano,Gemma-4-E4B-IT,Qwen3.5-4B
prompt_name,,,
short_coding,Here's a quick Python function that tells whet...,Here are a few ways to write a Python function...,Here's an efficient Python function to check i...
short_translation,<empty>,"There are a few ways to translate this, depend...",Thinking Process: 1. **Analyze the Request:** ...
short_factual,"The user asks: ""What are the three laws of the...",The first law states that energy cannot be cre...,Thinking Process: 1. **Analyze the Request:** ...
medium_coding,We need to write Python class called CSVValida...,"```python import csv from typing import List, ...",Here's a thinking process that leads to the su...
medium_analysis,We need to compare Transformer vs Flash Attent...,## Comparison of Original Transformer Attentio...,Here's a thinking process that leads to the co...
medium_creative,**Prompt Optimizer - Product Brief** *Problem ...,## Product Brief: Prompt Optimizer (AI Enhance...,Thinking Process: 1. **Analyze the Request:** ...
long_coding,We need to design Python module token_budget_m...,This is a comprehensive implementation that me...,Here's a thinking process that leads to the su...
long_analysis,**Literature Review - Prompt-Compression Techn...,## Literature Review: Prompt Compression Techn...,Here's a thinking process that leads to the su...


Completion tokens and throughput


model_name,GPT-OSS-Nano,Gemma-4-E4B-IT,Qwen3.5-4B
prompt_name,,,
short_coding,256 tok | 13.62 tok/s,256 tok | 13.76 tok/s,256 tok | 21.20 tok/s
short_translation,ERROR: LM Studio text generation failed for mo...,256 tok | 13.85 tok/s,256 tok | 22.60 tok/s
short_factual,256 tok | 13.60 tok/s,58 tok | 12.75 tok/s,256 tok | 22.05 tok/s
medium_coding,512 tok | 13.67 tok/s,512 tok | 13.41 tok/s,512 tok | 21.56 tok/s
medium_analysis,512 tok | 13.76 tok/s,512 tok | 13.43 tok/s,512 tok | 20.33 tok/s
medium_creative,512 tok | 13.65 tok/s,205 tok | 13.13 tok/s,512 tok | 16.32 tok/s
long_coding,1024 tok | 13.64 tok/s,1024 tok | 13.14 tok/s,1024 tok | 11.88 tok/s
long_analysis,1024 tok | 16.04 tok/s,1024 tok | 12.72 tok/s,1024 tok | 12.42 tok/s


In [4]:
# Cell 4 - Latency and throughput comparison
ALL_RESULTS = ensure_results_loaded()
LATENCY_SUMMARY_DF = build_latency_summary_df(ALL_RESULTS)
print("Overall repeated-run latency summary")
display(LATENCY_SUMMARY_DF)

PROMPT_LENGTH_LATENCY_DF = build_prompt_length_latency_df(ALL_RESULTS)
if PROMPT_LENGTH_LATENCY_DF.empty:
    print("No prompt-length latency comparison is available.")
else:
    print("Latency comparison grouped by prompt length")
    display(PROMPT_LENGTH_LATENCY_DF)

Overall repeated-run latency summary


,model_name,available,precheck_ready,avg_latency_ms,std_latency_ms,avg_tokens_per_second
0,Qwen3.5-4B,True,True,7661.10,133.85,12.54
1,Gemma-4-E4B-IT,True,True,2304.60,31.39,11.28
2,GPT-OSS-Nano,True,True,6364.74,254.38,15.11


Latency comparison grouped by prompt length


,prompt_length,model_name,avg_latency_ms,std_latency_ms,avg_tokens_per_second
0,long,GPT-OSS-Nano,69472.20,7949.49,14.84
1,long,Gemma-4-E4B-IT,79202.74,1809.24,12.93
2,long,Qwen3.5-4B,84339.83,2656.17,12.15
3,medium,GPT-OSS-Nano,37387.09,154.69,13.69
4,medium,Gemma-4-E4B-IT,30642.61,13021.00,13.32
5,medium,Qwen3.5-4B,26768.31,4052.86,19.40
6,short,GPT-OSS-Nano,13606.54,9004.96,9.07
7,short,Gemma-4-E4B-IT,13877.49,8080.17,13.45
8,short,Qwen3.5-4B,11669.46,377.21,21.95


In [5]:
# Cell 5 - Structured output success rate
ALL_RESULTS = ensure_results_loaded()
STRUCTURED_DF = build_structured_results_df(ALL_RESULTS)
STRUCTURED_SUCCESS_RATE_DF = (
    STRUCTURED_DF[["model_name", "structured_success"]]
    .assign(structured_success_rate=lambda df: df["structured_success"].astype(float))
    .drop(columns=["structured_success"])
)

display(STRUCTURED_DF)
print("Structured success rate")
display(STRUCTURED_SUCCESS_RATE_DF)

,model_name,available,precheck_ready,structured_success,returned_content,error
0,Qwen3.5-4B,True,True,True,"{""intent"": ""prompt optimization"", ""keywords"": ...",
1,Gemma-4-E4B-IT,True,True,True,"{""intent"": ""Prompt Optimization"", ""keywords"": ...",
2,GPT-OSS-Nano,True,True,True,"{""intent"": ""Intent"", ""keywords"": [""sh"", ""..."", ""...",


Structured success rate


,model_name,structured_success_rate
0,Qwen3.5-4B,1.0
1,Gemma-4-E4B-IT,1.0
2,GPT-OSS-Nano,1.0


In [6]:
# Cell 6 - Composite score and recommendation placeholder
ALL_RESULTS = ensure_results_loaded()
SUMMARY_DF = build_run_summary_df(ALL_RESULTS).copy()

if not SUMMARY_DF.empty:
    SUMMARY_DF["weighted_score_placeholder"] = pd.NA
    SUMMARY_DF = SUMMARY_DF.sort_values(
        by=["available", "precheck_ready", "structured_success_rate", "response_non_empty_rate", "avg_tokens_per_second"],
        ascending=[False, False, False, False, False],
        na_position="last",
    ).reset_index(drop=True)

display(SUMMARY_DF)

print("Latest saved results")
print(LATEST_RESULTS_PATH)
print()
print("Weighted score formula placeholder")
print("SUMMARY_DF['weighted_score_placeholder'] = <Human fills weights here>")
print()
print("Recommendation placeholder")
print("- TODO: Human reviews precheck stability, quality, latency, and structured output before selecting the final local model.")

,model_name,actual_model_id,available,precheck_ready,avg_latency_ms,avg_tokens_per_second,structured_success_rate,response_non_empty_rate,load_error,weighted_score_placeholder
0,Qwen3.5-4B,qwen3.5-4b,True,True,7661.10,12.54,1.0,1.000,,<NA>
1,Gemma-4-E4B-IT,gemma-4-e4b-it,True,True,2304.60,11.28,1.0,1.000,,<NA>
2,GPT-OSS-Nano,gpt-oss-nano,True,True,6364.74,15.11,1.0,0.875,,<NA>


Latest saved results
/Users/corey/Desktop/JHU/Generative AI/TokenLess/artifacts/benchmarks/02_model_benchmark_latest.json

Weighted score formula placeholder
SUMMARY_DF['weighted_score_placeholder'] = <Human fills weights here>

Recommendation placeholder
- TODO: Human reviews precheck stability, quality, latency, and structured output before selecting the final local model.
